# Модуль 16.6.5 — практикум: рой на Multica, староста и помощники

Домашка к модулю 16.6.5 «Практикум на Multica: староста и помощники» — лекция на сайте курса.

**Этот ноутбук запускается локально, на своей машине. Бейджа Colab у него нет и не будет.**
Задание про локальную платформу-фабрику (Multica на вашем Docker) и локальный мир игры (чекаут
движка Cognopolis со своей базой). Ни того, ни другого в облачном ноутбуке нет, поэтому Colab и
Kaggle тут не помощники — врать бейджем не будем.

Ноутбук — инструкция плюс проверка. Он не запускает рой за вас: рой поднимают `scripts/setup.sh` и
одна команда `multica issue create`. Ноутбук делает другое — читает **факты**: что на самом деле
лежит на складе поселения (по MCP, мимо отчётов агентов) и как на самом деле выглядит дерево задач
в платформе (через `multica ... --output json`). Дальше он сверяет факты с ожиданием и печатает
табличку критериев приёма.

## Два трека

| Трек | Кому | Что нужно | Что делаете |
| --- | --- | --- | --- |
| **A** | всем | Python 3 и чекаут заготовки | разбираете заготовку по файлам, считаете план роя тем же алгоритмом, что и староста, проектируете расширение на бумаге |
| **B** | у кого поднята Multica | плюс платформа, чекаут движка и под-токен жителя | всё из трека A плюс живой прогон: четвёртый помощник, смена цели, намеренная поломка |

Трек A проходит **целиком**: ячейки, которым нужна платформа или движок, печатают мягкий пропуск и
не падают. Ключей от внешних API тут нет вообще — ни одного обращения в сеть.

## Что вы унесёте

- карту заготовки: где инструкции, где знание, где секреты и что будит агента;
- планировщик старосты в двадцать строк на Python — тот же разбор цели по рецептам с вычитанием
  склада, который делает `elder-yoda`, только руками и наглядно;
- вывод проверочных ячеек: `stored` из мира, дерево задач из платформы и разбор «кто кого будил»;
- заполненную табличку критериев приёма — её и сдаёте.

## Ссылки

- Заготовка роя: `github.com/ITrubnikov/Train_of_Thought-Cognopolis-demo-multica_V2` — репозиторий
  приватный, доступ у автора курса.
- Тот же рой без платформы (к модулю 16.7):
  `github.com/ITrubnikov/Train_of_Thought-Cognopolis-demo-multu` — тоже приватный.
- Эта домашка:
  [notebooks/module-16-6-5-multica-swarm](https://github.com/ITrubnikov/Train_of_Thought-homework/tree/main/notebooks/module-16-6-5-multica-swarm)

## Шаг 0. Настройки: пути и токен

Ноутбук ничего не зашивает в код. Всё, что он знает про вашу машину, приезжает из переменных
окружения — тех же самых, что лежат в `.env` заготовки:

| Переменная | Что это | Кому нужна |
| --- | --- | --- |
| `SWARM_REPO` | путь к чекауту заготовки роя | Задача 1, оба трека |
| `COGNOPOLIS_ENGINE` | путь к чекауту движка игры | ячейки про мир, трек B |
| `COGNOPOLIS_PYTHON` | python из venv движка (по умолчанию `<движок>/.venv/bin/python`) | то же |
| `DATABASE_URL` | база мира; для SQLite путь абсолютный, четыре слэша | то же |
| `COGNOPOLIS_TOKEN` | под-токен любого жителя `demoswarm` | то же |
| `MULTICA_WORKSPACE_ID` | id воркспейса роя, если воркспейсов несколько | ячейки про очередь задач |

Задать их проще всего до запуска Jupyter:

```bash
export SWARM_REPO=~/src/Train_of_Thought-Cognopolis-demo-multica_V2
export COGNOPOLIS_ENGINE=~/src/Train_of_Thought-Cognopolis
export DATABASE_URL="sqlite+aiosqlite:////absolute/path/to/cognopolis.db"   # четыре слэша
export COGNOPOLIS_TOKEN=...        # или оставьте пустым и включите ASK_TOKEN ниже
jupyter lab notebook.ipynb
```

Про токен отдельно. Текстом в ячейку он не пишется — никогда. Либо переменная окружения, либо
`getpass` (`ASK_TOKEN = True` — тогда ноутбук спросит токен при выполнении ячейки и в файл его не
сохранит). По умолчанию `ASK_TOKEN = False`, чтобы `Run all` не замирал на вводе. Значение токена
ноутбук не печатает — ни целиком, ни куском.

In [ ]:
# Шаг 0 — настройки. Только стандартная библиотека, ни одного pip install.
import getpass
import hashlib
import json
import os
import pathlib
import re
import shutil
import subprocess

ASK_TOKEN = False          # True — спросить под-токен через getpass прямо здесь

UNSET = pathlib.Path("__не_задано__")      # заведомо несуществующий путь вместо пустой строки


def env_path(name, default=UNSET):
    """Пустая переменная окружения — это «не задано», а не текущий каталог."""
    raw = os.environ.get(name, "").strip()
    return pathlib.Path(raw).expanduser() if raw else default


def where(path):
    return "— не задан" if path == UNSET else str(path)


SWARM_REPO = env_path("SWARM_REPO")
ENGINE = env_path("COGNOPOLIS_ENGINE")
ENGINE_PY = env_path("COGNOPOLIS_PYTHON",
                     (ENGINE / ".venv" / "bin" / "python") if ENGINE != UNSET else UNSET)
DATABASE_URL = os.environ.get("DATABASE_URL", "")
WORKSPACE_ID = os.environ.get("MULTICA_WORKSPACE_ID", "")

TOKEN = os.environ.get("COGNOPOLIS_TOKEN", "").strip()
if not TOKEN and ASK_TOKEN:
    TOKEN = getpass.getpass("под-токен жителя demoswarm (в ноутбук не попадёт): ").strip()

NB_PATH = pathlib.Path("notebook.ipynb")   # для самопроверки «токена нет в сдаваемом файле»


def show(name, value, ok):
    print(f"  {'есть' if ok else 'нет '}  {name:<18} {value}")


print("Настройки (значение токена не печатается никогда):")
show("SWARM_REPO", where(SWARM_REPO), SWARM_REPO.is_dir())
show("COGNOPOLIS_ENGINE", where(ENGINE), ENGINE.is_dir())
show("COGNOPOLIS_PYTHON", where(ENGINE_PY), ENGINE_PY.is_file())
show("DATABASE_URL", DATABASE_URL or "— не задан", DATABASE_URL.startswith("sqlite"))
show("COGNOPOLIS_TOKEN", "задан" if TOKEN else "— не задан", bool(TOKEN))
show("MULTICA_WORKSPACE", WORKSPACE_ID or "активный воркспейс CLI", True)
print("\nЧего нет — тем ячейкам будет мягкий пропуск. Трек A проходит и без них.")

## Шаг 1. Префлайт: жив ли рантайм

Первый вопрос в мультиагентной системе — не «что не так с промптом», а **жив ли рантайм**. На
сборке заготовки это стоило часа: агенты стартовали и молча висели — процесс жив, ноль процентов
CPU, ни одного сетевого соединения, задача вечно в `running`. Ни инструкции, ни MCP там были ни при
чём, лечилось `multica daemon restart`.

Поэтому проверка идёт снизу вверх: CLI в `PATH`, авторизация, демон, runtime, чекаут движка, его
python, файл базы мира, чекаут заготовки. Ячейка ничего не чинит и не падает: печатает, что есть, и
говорит, что делать с тем, чего нет.

In [ ]:
# Шаг 1 — префлайт. Ничего не меняет: только смотрит и печатает.
def run(args, timeout=25):
    """Запустить команду и вернуть (rc, stdout, stderr). Наружу — только текст, без исключений."""
    try:
        p = subprocess.run(args, capture_output=True, text=True, timeout=timeout)
        return p.returncode, p.stdout.strip(), p.stderr.strip()
    except FileNotFoundError:
        return 127, "", f"{args[0]}: не найден в PATH"
    except subprocess.TimeoutExpired:
        return 124, "", f"{' '.join(args)}: таймаут {timeout} с"


def mc(*args, timeout=40):
    """multica с опциональным --workspace-id (флаг глобальный, идёт перед подкомандой)."""
    head = ["multica"] + (["--workspace-id", WORKSPACE_ID] if WORKSPACE_ID else [])
    return run(head + list(args), timeout=timeout)


def mc_json(*args, timeout=40):
    """multica ... --output json → разобранный объект или None, если не получилось."""
    rc, out, _err = mc(*args, "--output", "json", timeout=timeout)
    if rc != 0 or not out:
        return None
    try:
        return json.loads(out)
    except json.JSONDecodeError:
        return None


checks = []


def check(name, ok, detail, fix):
    checks.append((name, bool(ok), detail, fix))


check("multica в PATH", shutil.which("multica") is not None,
      shutil.which("multica") or "не найден",
      "поставьте CLI Multica и выполните multica login")

rc, out, err = mc("auth", "status", timeout=20)
check("авторизация CLI", rc == 0,
      (out.splitlines() or [err or "нет ответа"])[0],
      "multica login; если сервер молчит — поднимите Docker-стек Multica")

rc, out, err = run(["multica", "daemon", "status"], timeout=20)
check("демон платформы", rc == 0 and "running" in out.lower(),
      (out.splitlines() or [err or "нет ответа"])[0],
      "multica daemon start; а если демон 'running', но агенты стоят — multica daemon restart")

rt = mc_json("runtime", "list", timeout=30)
rt_rows = rt if isinstance(rt, list) else (rt or {}).get("runtimes", [])
check("runtime online", any(r.get("status") == "online" for r in (rt_rows or [])),
      ", ".join(f"{r.get('name')} [{r.get('status')}]" for r in (rt_rows or [])) or "не прочитан",
      "runtime offline — сначала Docker, потом multica daemon start")

check("чекаут движка", ENGINE.is_dir() and (ENGINE / "server" / "mcp_server.py").is_file(),
      where(ENGINE),
      "склонируйте движок игры и задайте COGNOPOLIS_ENGINE")

check("python движка", ENGINE_PY.is_file() and os.access(ENGINE_PY, os.X_OK),
      where(ENGINE_PY),
      "нужен python из venv движка: голому питону не видны fastapi, sqlmodel и mcp")

db_path = ""
if DATABASE_URL.startswith("sqlite"):
    m = re.search(r"sqlite(?:\+\w+)?://(/+.*)$", DATABASE_URL)
    if m:
        db_path = "/" + m.group(1).lstrip("/").split("?")[0]
check("база мира", bool(db_path) and pathlib.Path(db_path).is_file(),
      db_path or DATABASE_URL or "не задан",
      "для SQLite путь обязан быть абсолютным (четыре слэша): относительный молча создаст пустой мир")

check("чекаут заготовки", SWARM_REPO.is_dir() and (SWARM_REPO / "agents").is_dir(),
      where(SWARM_REPO),
      "нужен для Задачи 1; репозиторий приватный, доступ у автора курса")

print(f"{'проверка':<20} {'итог':<6} что видно")
print("-" * 96)
for name, ok, detail, _fix in checks:
    print(f"{name:<20} {'ok' if ok else 'нет':<6} {detail}")

print("\nЧто делать с тем, чего нет:")
missing = [(n, fix) for n, ok, _d, fix in checks if not ok]
for name, fix in missing:
    print(f"  {name}: {fix}")
if not missing:
    print("  ничего, всё на месте")

HAVE_MULTICA = checks[0][1] and checks[1][1]
HAVE_ENGINE = checks[4][1] and checks[5][1] and checks[6][1] and bool(TOKEN)
HAVE_SWARM_REPO = checks[7][1]
print(f"\nФлаги: платформа={HAVE_MULTICA}, мир по MCP={HAVE_ENGINE}, заготовка={HAVE_SWARM_REPO}")
print("Трек A: дальше всё считается и с тремя False — на слепке мира из урока.")

## Задача 1. Разобрать заготовку по файлам

Первое, что надо уметь про чужую мультиагентную систему, — сказать, **где что живёт**. Не «там
где-то промпты», а конкретный файл. В заготовке четыре сорта содержимого, и они разложены по разным
местам намеренно:

- **инструкции** — кто ты и какова граница твоей роли (`agents/`);
- **знание** — правила мира и протокол очереди, одинаковые для всех, кому они нужны (`skills/`);
- **секреты** — токены, которых в репозитории нет вовсе: только плейсхолдеры и механика доставки;
- **будильники** — то, от чего агент вообще просыпается. Это не файл, это набор событий очереди.

Ниже шесть вопросов. Отвечать надо **путём к файлу** относительно корня заготовки. Ячейка проверяет
честно: открывает файл и ищет в нём маркер. Маркер показан в подсказке — по нему же удобно искать
`grep`-ом, если застряли. Ответ засчитан, только если файл существует и маркер в нём есть.

Седьмой вопрос — про будильники: отметьте, какие события реально будят агента. Правильный ответ
ноутбук держит хешем, подсмотреть в коде не выйдет; при ошибке он скажет, какую секцию перечитать.

In [ ]:
# Задача 1 — карта заготовки. Заполните ANSWERS и WAKES, запустите ячейку.
QUESTIONS = {
    "instructions_elder": (
        "Где написана граница роли старосты: что он делает сам, а чего не трогает руками?",
        r"Не твоё"),
    "instructions_worker": (
        "В каком файле лежат инструкции сразу для троих помощников?",
        r"сразу для троих"),
    "knowledge_world": (
        "Где живёт знание про мир: две зоны хранения, разгрузка домом, коды ошибок?",
        r"Две зоны хранения"),
    "knowledge_queue": (
        "Где написано, что будит агента, а что нет?",
        r"Что будит агента"),
    "secrets_template": (
        "Где лежат плейсхолдеры секретов — и ни одного настоящего токена?",
        r"TOKEN_YODA"),
    "secrets_delivery": (
        "Какой файл доставляет токены в custom_env агента, не пропуская их через argv?",
        r"--custom-env-file"),
}

# ваши ответы: путь относительно корня заготовки, например "agents/elder.md"
ANSWERS = {
    "instructions_elder": "",
    "instructions_worker": "",
    "knowledge_world": "",
    "knowledge_queue": "",
    "secrets_template": "",
    "secrets_delivery": "",
}

WAKE_EVENTS = [
    "создание issue с --assignee и активным статусом (todo)",
    "комментарий с mention агента, написанный другим агентом",
    "перевод issue из backlog в активный статус при выставленном assignee",
    "закрытие последней под-задачи ступени (у под-задач есть --parent и --stage)",
    "обычный комментарий без смены статуса",
    "multica issue rerun <id>",
]
# ваш ответ: True — будит, False — не будит. По одному значению на каждую строку выше.
WAKES = [None, None, None, None, None, None]

WAKES_EXPECTED_SHA = "e13733d9305592cb21939f9525f2e216866b2d85cbf12c1ea3f3fce754e47997"


def check_answer(key):
    """Вернуть (вердикт, подсказка) по одному ответу."""
    question, marker = QUESTIONS[key]
    answer = (ANSWERS.get(key) or "").strip()
    if answer.startswith("./"):        # именно префикс, а не lstrip: ".env-example" не трогаем
        answer = answer[2:]
    if not answer:
        return "не отвечено", f"ищите файл, в котором есть: {marker}"
    path = SWARM_REPO / answer
    if not path.is_file():
        return "нет файла", f"проверьте путь; маркер нужного файла: {marker}"
    text = path.read_text(encoding="utf-8", errors="replace")
    if not re.search(marker, text):
        return "файл не тот", f"в нём нет маркера: {marker}"
    return "ok", ""


task1_files_ok = 0
print("Карта заготовки:\n")
if not HAVE_SWARM_REPO:
    print("  пропуск: чекаут заготовки не найден (SWARM_REPO). Репозиторий приватный,")
    print("  доступ у автора курса. Остальные ячейки от него не зависят.")
else:
    for key in QUESTIONS:
        verdict, hint = check_answer(key)
        task1_files_ok += 1 if verdict == "ok" else 0
        print(f"  [{verdict:^12}] {QUESTIONS[key][0]}")
        print(f"                 ответ: {ANSWERS.get(key) or '—'}")
        if hint:
            print(f"                 подсказка: {hint}")
    print(f"\n  файлов найдено верно: {task1_files_ok} из {len(QUESTIONS)}")

print("\nЧто будит агента:")
if any(w is None for w in WAKES):
    task1_wakes_ok = False
    print("  пропуск: заполните список WAKES значениями True и False")
else:
    vector = "".join("1" if bool(w) else "0" for w in WAKES)
    task1_wakes_ok = hashlib.sha256(vector.encode()).hexdigest() == WAKES_EXPECTED_SHA
    for event, flag in zip(WAKE_EVENTS, WAKES):
        print(f"  {'будит   ' if flag else 'не будит'}  {event}")
    print(f"\n  вердикт: {'ok' if task1_wakes_ok else 'не сходится'}")
    if not task1_wakes_ok:
        print("  перечитайте секцию «Что будит агента, а что нет» в скилле multica-issue-protocol")

TASK1_OK = (task1_files_ok == len(QUESTIONS)) and task1_wakes_ok
print(f"\nЗадача 1: {'зачтена' if TASK1_OK else 'не зачтена'}")

## Шаг 2. Мир по MCP: тонкий клиент на одной стандартной библиотеке

Мир игры отвечает только по MCP и только по stdio — HTTP-транспорта у этого сервера нет. Клиент для
него помещается в тридцать строк: поднять подпроцесс `python -m server.mcp_server`, отправить
`initialize`, следом нотификацию `notifications/initialized`, дальше слать `tools/call` и читать
построчный JSON-RPC 2.0 из stdout. SDK не нужен — это ровно тот протокол, который вы разбирали в
модуле 14.5, только теперь на голом `subprocess`.

Две детали, на которых легко споткнуться:

- **токен идёт аргументом тула** (`observe(token=...)`), а не заголовком: заголовков в MCP нет;
- **отказ приезжает обычным результатом**, а не исключением транспорта: в теле ответа лежит
  `{"error": {"code": ..., "message": ...}}`. Поэтому после каждого вызова смотрим ключ `error`.

Чтение (`observe`, `get_recipes`) кулдаун не тратит и мир не меняет — ячейка безопасна даже посреди
работы роя.

In [ ]:
# Шаг 2 — MCP-клиент по stdio. Стандартная библиотека: subprocess, json, threading, queue.
import queue
import threading


class McpStdio:
    """Минимальный MCP-клиент: один подпроцесс, построчный JSON-RPC 2.0, жёсткий таймаут."""

    def __init__(self, python, engine, database_url, timeout=60):
        env = dict(os.environ, PYTHONPATH=str(engine), DATABASE_URL=database_url)
        self.timeout = timeout
        self.proc = subprocess.Popen(
            [str(python), "-m", "server.mcp_server"],
            stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL,
            env=env, text=True, bufsize=1, cwd=str(engine),
        )
        self._lines = queue.Queue()
        threading.Thread(target=self._pump, daemon=True).start()
        self._id = 0
        self.call("initialize", {
            "protocolVersion": "2025-06-18",
            "capabilities": {},
            "clientInfo": {"name": "homework-16-6-5", "version": "1"},
        })
        self._write({"jsonrpc": "2.0", "method": "notifications/initialized", "params": {}})

    def _pump(self):
        for line in self.proc.stdout:
            self._lines.put(line)
        self._lines.put(None)

    def _write(self, obj):
        self.proc.stdin.write(json.dumps(obj) + "\n")
        self.proc.stdin.flush()

    def call(self, method, params=None):
        self._id += 1
        my_id = self._id
        self._write({"jsonrpc": "2.0", "id": my_id, "method": method, "params": params or {}})
        while True:
            line = self._lines.get(timeout=self.timeout)
            if line is None:
                raise RuntimeError("MCP-сервер закрыл поток: проверьте python движка и DATABASE_URL")
            msg = json.loads(line)
            if msg.get("id") == my_id:
                if "error" in msg:
                    raise RuntimeError(f"JSON-RPC error: {msg['error']}")
                return msg.get("result", {})

    def tool(self, name, **arguments):
        """Вызвать тул и вернуть разобранный ответ. Игровой отказ приезжает ключом error."""
        result = self.call("tools/call", {"name": name, "arguments": arguments})
        text = (result.get("content") or [{}])[0].get("text", "{}")
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            return {"raw": text}

    def close(self):
        try:
            self.proc.stdin.close()
        finally:
            self.proc.terminate()


MCP = None
if HAVE_ENGINE:
    MCP = McpStdio(ENGINE_PY, ENGINE, DATABASE_URL)
    tools = [t["name"] for t in MCP.call("tools/list").get("tools", [])]
    needed = ["observe", "get_character", "get_roster", "get_recipes", "get_items",
              "get_map", "move", "gather", "craft", "equip"]
    print(f"MCP поднят: тулов у движка {len(tools)}")
    print("нужные этому рою:", ", ".join(t for t in needed if t in tools))
    print("не хватает:", ", ".join(t for t in needed if t not in tools) or "ничего")
else:
    print("пропуск: нет чекаута движка, его python или под-токена — MCP не поднимаем.")
    print("Дальше ноутбук возьмёт слепок мира из урока, и трек A пройдёт целиком.")

## Шаг 3. Мир: ожидание против факта

Главный инвариант заготовки: **староста принимает работу по миру, а не по комментариям помощника**.
Помощник может честно добыть девять брёвен, честно увидеть их в рюкзаке, честно написать «добыл 9» —
и не дойти до дома. Отдельного `deposit` в игре нет: разгрузка это побочный эффект шага на домашний
тайл `(0,0)`. Пока житель не дома, для склада работы не было, и крафтер через минуту получит
`not_enough_resources`.

Поэтому проверочная ячейка читает `observe(token).character.stored` — тот самый общий склад
поселения — и печатает таблицу «ожидание против факта» для цели.

Если движка под рукой нет, ячейка берёт **слепок из урока**: рецепты и склад сразу после сида
(`{axe: 1, pickaxe: 1, goblin_ear: 3, wolf_pelt: 1}` — дерева и камня нет намеренно, иначе рою
нечего делать). Слепок помечается в выводе: план на нём считается тот же, но это не живой мир.

In [ ]:
# Шаг 3 — склад и рецепты: живой мир или слепок из урока.
# Слепок соответствует состоянию сразу после seed-world.sh (см. лекцию 16.6.5).
SNAPSHOT_RECIPES = {
    "axe_handle": {"inputs": {"wood": 3}, "building": "sawmill", "building_level_req": 1},
    "axe": {"inputs": {"axe_handle": 1, "stone": 2}, "building": "sawmill", "building_level_req": 1},
    "pickaxe": {"inputs": {"axe_handle": 1, "stone": 4}, "building": "sawmill", "building_level_req": 1},
    "spear": {"inputs": {"axe_handle": 1, "stone": 3}, "building": "forge", "building_level_req": 1},
    "copper_ingot": {"inputs": {"copper": 3}, "building": "forge", "building_level_req": 1},
}
SNAPSHOT_STORED = {"axe": 1, "pickaxe": 1, "goblin_ear": 3, "wolf_pelt": 1}

GOAL_ITEM, GOAL_QTY = "spear", 3      # цель демо: «сделать 3 копья»

RECIPES, STORED, BUILDINGS = SNAPSHOT_RECIPES, SNAPSHOT_STORED, []
WORLD_SOURCE = "слепок из урока (мир сразу после сида)"

if MCP is not None:
    live_recipes = {r["recipe"]: r for r in MCP.tool("get_recipes").get("recipes", [])}
    snap = MCP.tool("observe", token=TOKEN)
    if "error" in snap:
        print("отказ движка:", snap["error"])
        print("invalid_token обычно значит пересев мира с нуля (тогда токены выданы заново)")
        print("или DATABASE_URL, который смотрит в другую базу. Считаем по слепку.\n")
    else:
        character = snap["character"]
        RECIPES = live_recipes or SNAPSHOT_RECIPES
        STORED = character.get("stored", {})
        BUILDINGS = character.get("buildings", [])
        WORLD_SOURCE = f"живой мир по MCP, житель {character.get('name')} ({character.get('role')})"


def plan_for(goal_item, goal_qty, recipes, stored):
    """Разбор цели, как его делает староста: развернуть дерево рецептов сверху вниз, на каждом
    узле вычесть то, что уже лежит на складе, недостачу сложить снизу вверх. Склад по дороге
    «резервируем», чтобы одна и та же позиция не засчиталась дважды."""
    left = dict(stored)
    crafts = []

    def need(item, qty):
        take = min(left.get(item, 0), qty)
        left[item] = left.get(item, 0) - take
        qty -= take
        if qty <= 0:
            return {}
        recipe = recipes.get(item)
        if recipe is None:                       # лист дерева — это сырьё, его добывают
            return {item: qty}
        raw = {}
        for src, per in recipe["inputs"].items():
            for key, value in need(src, per * qty).items():
                raw[key] = raw.get(key, 0) + value
        crafts.append((item, qty))               # порядок переделов выпадает сам, руками не задаём
        return raw

    return need(goal_item, goal_qty), crafts


raw, crafts = plan_for(GOAL_ITEM, GOAL_QTY, RECIPES, STORED)

print(f"Источник данных: {WORLD_SOURCE}")
print(f"Цель: {GOAL_QTY} x {GOAL_ITEM}\n")
print(f"{'позиция':<14} {'ожидание':>9} {'факт':>6}  вердикт")
print("-" * 56)
have_goal = STORED.get(GOAL_ITEM, 0)
print(f"{GOAL_ITEM:<14} {GOAL_QTY:>9} {have_goal:>6}  "
      f"{'цель закрыта' if have_goal >= GOAL_QTY else 'ещё нет'}")
for item, qty in sorted(raw.items()):
    print(f"{item:<14} {qty:>9} {STORED.get(item, 0):>6}  добыть (сырьё)")
if not raw:
    print("(сырьё добывать не нужно — склад покрывает цель)")

print("\nПлан переделов, в порядке исполнения:")
for item, qty in crafts:
    where = (RECIPES.get(item) or {}).get("building", "?")
    print(f"  craft {item} x{qty}  на станции {where}")
if not crafts:
    print("  пусто: план схлопнулся по складу, наряды не нужны")

if BUILDINGS:
    print("\nЗдания поселения:", ", ".join(f"{b['kind']} lvl{b['level']}" for b in BUILDINGS))
print("\nСклад целиком:", json.dumps(STORED, ensure_ascii=False, sort_keys=True))
print("Напоминание: инструмент даёт бонус только надетым, поэтому axe и pickaxe могут")
print("не лежать на складе — они на ком-то из жителей.")

## Шаг 4. Дерево задач: ступени, статусы и кто кого будил

Второе общее состояние роя — очередь задач платформы. В ней видно то, чего нет в мире: кто что взял,
в каком порядке и что кого разбудило.

Ячейка читает `multica issue list --output json` (это объект `{has_more, issues}`, а не голый
массив — форму вывода стоит проверять, а не угадывать), для каждой корневой цели дёргает
`multica issue children <id> --output json` (а он отдаёт `{stages: [...]}` уже со счётчиками
`done`/`total`) и раскладывает комментарии по авторам.

«Кто кого будил» читается прямо из комментариев: у системного stage-уведомления `author_type` равен
`system`, у отчёта помощника — `agent`. Это будильник, который видно постфактум. Остальные три
будильника следов в ленте не оставляют, поэтому ячейка допечатывает их списком — сверяйте дерево с
полным набором, а не только с тем, что попало в комментарии.

In [ ]:
# Шаг 4 — дерево задач и разбор пробуждений.
GOAL_ISSUE = ""     # например "COG-5"; пусто — взять самую свежую корневую цель с под-задачами

AGENT_NAMES = {}
TREE_ROOT = None

if not HAVE_MULTICA:
    print("пропуск: нет CLI Multica или авторизации — очередь задач не читаем (это трек B).")
else:
    agents = mc_json("agent", "list") or []
    AGENT_NAMES = {a["id"]: a["name"] for a in agents}
    print(f"Агенты воркспейса ({len(agents)}):")
    for a in agents:
        skills = ", ".join(s.get("name", "") for s in (a.get("skills") or []))
        print(f"  {a['name']:<14} {a.get('model', '?'):<28} скиллы: {skills}")
        print(f"  {'':<14} custom_env: ключей {a.get('custom_env_key_count', 0)}, "
              f"значения CLI не отдаёт")

    listing = mc_json("issue", "list", "--limit", "100")
    issues = listing.get("issues", []) if isinstance(listing, dict) else (listing or [])
    roots = sorted((i for i in issues if not i.get("parent_issue_id")),
                   key=lambda i: i.get("created_at", ""), reverse=True)

    root = None
    if GOAL_ISSUE:
        root = next((i for i in issues if i.get("identifier") == GOAL_ISSUE), None)
        if root is None:
            print(f"\nзадача {GOAL_ISSUE} в этом воркспейсе не найдена")
    if root is None:
        for candidate in roots:
            if (mc_json("issue", "children", candidate["id"]) or {}).get("stages"):
                root = candidate
                break
        root = root or (roots[0] if roots else None)

    if root is None:
        print("\nв воркспейсе нет ни одной задачи — поставьте цель рою:")
        print('  multica issue create --title "Сделать 3 копья" --assignee elder-yoda --status todo')
    else:
        TREE_ROOT = root
        print(f"\nЦель: {root['identifier']} [{root['status']}] {root['title']}")
        print(f"  исполнитель: {AGENT_NAMES.get(root.get('assignee_id'), '—')}, "
              f"создана {root.get('created_at')}, обновлена {root.get('updated_at')}")

        stages = (mc_json("issue", "children", root["id"]) or {}).get("stages", [])
        if not stages:
            print("\n  под-задач нет: либо цель ещё не разобрана, либо план схлопнулся по складу")
        for stage in stages:
            print(f"\n  ступень {stage['stage']} — {stage['done']}/{stage['total']} закрыто")
            for issue in stage["issues"]:
                name = AGENT_NAMES.get(issue.get("assignee_id"), "—")
                print(f"    {issue['identifier']:<7} [{issue['status']:<11}] {name:<14} "
                      f"{issue['title']}")

        print("\nКто кого будил (по следам в комментариях):")
        keys = [root["identifier"]] + [i["identifier"] for s in stages for i in s["issues"]]
        for key in keys:
            for c in (mc_json("issue", "comment", "list", key) or []):
                lines = (c.get("content") or "").strip().splitlines()
                head = lines[0][:74] if lines else ""
                if c.get("author_type") == "system":
                    print(f"  {key:<7} будильник платформы: stage-уведомление разбудило "
                          f"{AGENT_NAMES.get(root.get('assignee_id'), 'assignee родителя')}")
                else:
                    who = AGENT_NAMES.get(c.get("author_id"), c.get("author_type", "?"))
                    print(f"  {key:<7} след агента {who:<14} {head}")

        print("\nПолный список будильников (следов в ленте не оставляют, но работают):")
        print("  1. issue create --assignee <агент> --status todo")
        print("  2. перевод из backlog в активный статус при выставленном assignee")
        print("  3. закрытие последней под-задачи ступени -> stage-уведомление assignee родителя")
        print("  4. multica issue rerun <id>")
        print("  Не будит: mention агент в агента (антипетлевая защита) и обычный комментарий.")

## Задача 2. Добавить четвёртого помощника, не трогая `agents/worker.md`

Главный приём заготовки: у троих помощников **один файл инструкций**. Ни имени, ни специализации,
ни «своих» клеток карты в нём нет. Кто ты — решает под-токен в `custom_env` агента; что ты делаешь —
решает текст наряда. Значит четвёртый помощник добавляется выдачей токена, а не написанием
четвёртого промпта.

**Трек B — сделать.** Сначала нужен ещё один житель, и путей два, оба честные:

1. *Нанять жителя в игре, без пересева мира.* Наём — действие игрока (экран «Жители» в интерфейсе,
   под капотом `POST /account/characters` под сессией с CSRF). MCP-тула для найма нет, и агенту он
   не положен. Потолок ростера равен уровню ратуши: в демо она пятого уровня, а жителей четверо —
   место есть. Цена списывается со **склада** и растёт с размером ростера: базовая ставка
   `wood 20 + stone 10`, умноженная на текущее число жителей, то есть для пятого — 80 wood и
   40 stone. Под-токен нового жителя выдаётся сразу при найме, чужие токены при этом не протухают.
2. *Дописать имя в сид и пересеять мир.* Дёшево по ресурсам: обычный пересев поверх существующего
   мира старым жителям токены **сохраняет**, а новому выдаёт свой — в `.env` добавляется одна
   переменная. А вот пересев с нуля (`--fresh`, снос базы) перевыпускает токены всем: тогда
   обновляете весь блок `TOKEN_*` и заново гоняете `setup.sh`, иначе первый же прогон встретит
   `invalid_token`.

Дальше платформа: в `setup.sh` копируете блок `case` под новое имя агента (`$WORKER_MODEL`,
`agents/worker.md`, имя жителя, его токен, тот же набор скиллов), добавляете имя в список цикла, а
токен — в `require_env` и в `.env-example`. **Файл `agents/worker.md` не трогаете.** Прогон
`./scripts/setup.sh` — и у вас четвёртый помощник.

Проверка ниже не верит на слово: она сравнивает **sha256 инструкций** у всех агентов `worker-*`.
Совпали — текст действительно один на всех. Разошлись — где-то дописали «а этот занимается камнем»,
и приём заготовки сломан: через три итерации у вас будет четыре расходящихся файла вместо одного.

**Трек A — расписать.** Заполните `TASK2_PLAN` шагами и файлами, которые нужно тронуть (и чем
именно), и поставьте `TASK2_WORKER_MD_UNTOUCHED = True`, если в вашем плане `agents/worker.md`
остаётся нетронутым.

In [ ]:
# Задача 2 — четвёртый помощник. Проверка: один текст инструкций на всех помощников.
TASK2_PLAN = """
"""            # трек A: шаги и файлы — что правим, что не правим, откуда берётся токен

TASK2_WORKER_MD_UNTOUCHED = False

TASK2_OK_B = False
if not HAVE_MULTICA:
    print("пропуск: платформы нет — это шаг трека B.")
else:
    agents = mc_json("agent", "list") or []
    workers = [a for a in agents if a["name"].startswith("worker-")]
    others = [a for a in agents if not a["name"].startswith("worker-")]

    print(f"{'агент':<16} {'модель':<28} {'sha инструкций':<16} длина")
    print("-" * 76)
    for a in agents:
        digest = hashlib.sha256((a.get("instructions") or "").encode()).hexdigest()[:12]
        print(f"{a['name']:<16} {a.get('model', '?'):<28} {digest:<16} "
              f"{len(a.get('instructions') or '')}")

    digests = {hashlib.sha256((a.get("instructions") or "").encode()).hexdigest() for a in workers}
    same_text = len(digests) == 1 and len(workers) >= 3
    print(f"\nпомощников: {len(workers)}; различных текстов инструкций у них: {len(digests)}")
    print(f"один текст на всех помощников: {'да' if same_text else 'нет'}")
    if others:
        print("у старосты текст свой — так и должно быть: другая роль, другой playbook")
    print(f"четвёртый помощник добавлен: {'да' if len(workers) >= 4 else 'ещё нет'}")
    TASK2_OK_B = same_text and len(workers) >= 4

    print("\nСекреты: CLI отдаёт только число ключей custom_env, сами значения — нет.")
    for a in agents:
        print(f"  {a['name']:<16} ключей в custom_env: {a.get('custom_env_key_count', 0)}")

TASK2_OK_A = (bool(TASK2_PLAN.strip()) and TASK2_WORKER_MD_UNTOUCHED
              and all(marker in TASK2_PLAN for marker in (".env", "setup.sh")))
print(f"\nТрек A, план расширения: {'заполнен' if TASK2_OK_A else 'не заполнен'}")
if TASK2_PLAN.strip() and not TASK2_OK_A:
    print("  в плане должны быть названы и .env (откуда приезжает токен), и setup.sh (кто")
    print("  раскатывает агента), и стоять TASK2_WORKER_MD_UNTOUCHED = True")

## Задача 3. Поменять цель и увидеть, как план схлопывается по складу

Разница между «умным» и «наивным» старостой ровно в одном шаге: вычесть склад **до** того, как
разворачивать очередной узел дерева рецептов. Наивный считает по рецепту и гонит двоих за сырьём,
которое уже лежит на складе.

Проверяется это без роя: планировщик из шага 3 — тот же алгоритм. Ячейка ниже прогоняет его по
списку целей на текущем складе. Исходов у него три: полное схлопывание (цель уже покрыта складом),
«добыча не нужна, только переделы» (сырьё уже лежит) и полный разворот с нуля. На складе сразу
после сида в таблице видно два из трёх — этого и достаточно, чтобы увидеть разницу.

**Трек B.** Поставьте рою цель, которую склад уже покрывает, и убедитесь, что староста закрывает её
без единого наряда:

```bash
multica issue create --title "Сделать 3 копья" --assignee elder-yoda --status todo
```

Впишите ключ этой задачи в `TASK3_ISSUE`. Критерий машинный и жёсткий: у задачи **нет под-задач**, а
статус терминальный. Значит план схлопнулся на разборе, а не «помощники быстро сбегали».

In [ ]:
# Задача 3 — смена цели. Планировщик считает по живому складу (или по слепку из урока).
TASK3_ISSUE = ""     # трек B: ключ задачи, план которой схлопнулся, например "COG-9"

GOALS = [(GOAL_ITEM, GOAL_QTY), ("spear", 4), ("axe_handle", 2), ("pickaxe", 1)]

print(f"Склад ({WORLD_SOURCE}):")
print(f"  {json.dumps(STORED, ensure_ascii=False, sort_keys=True)}\n")
print(f"{'цель':<18} {'добыть':<24} {'крафтить':<28} исход")
print("-" * 96)
outcomes = set()
for item, qty in GOALS:
    raw_i, crafts_i = plan_for(item, qty, RECIPES, STORED)
    raw_s = ", ".join(f"{k} {v}" for k, v in sorted(raw_i.items())) or "—"
    crafts_s = " -> ".join(f"{c} x{q}" for c, q in crafts_i) or "—"
    if not crafts_i and not raw_i:
        verdict = "схлопнулось: склад покрывает цель"
    elif not raw_i:
        verdict = "добыча не нужна, только переделы"
    else:
        verdict = "полный разворот"
    outcomes.add(verdict)
    print(f"{str(qty) + ' x ' + item:<18} {raw_s:<24} {crafts_s:<28} {verdict}")

TASK3_PLANNER_OK = len(outcomes) >= 2
print("\nЧитается так: наряды рождаются из разницы «нужно минус лежит», а не из рецепта.")
print("Цель, которую склад уже покрывает, стоит рою один комментарий и ноль нарядов.")
print(f"Разных исходов в таблице: {len(outcomes)} "
      f"({'видно и схлопывание, и разворот' if TASK3_PLANNER_OK else 'добавьте цель другого типа'})")

TASK3_OK = False
if TASK3_ISSUE and HAVE_MULTICA:
    issue = mc_json("issue", "get", TASK3_ISSUE)
    if not issue:
        print(f"\n{TASK3_ISSUE}: не найдена")
    else:
        stages = (mc_json("issue", "children", TASK3_ISSUE) or {}).get("stages", [])
        n_kids = sum(len(s["issues"]) for s in stages)
        comments = mc_json("issue", "comment", "list", TASK3_ISSUE) or []
        by_agent = [c for c in comments if c.get("author_type") == "agent"]
        TASK3_OK = (n_kids == 0 and issue.get("status") in {"done", "cancelled"}
                    and bool(by_agent))
        print(f"\n{TASK3_ISSUE} [{issue.get('status')}] {issue.get('title')}")
        print(f"  под-задач: {n_kids}, комментариев от агентов: {len(by_agent)}")
        if by_agent:
            head = (by_agent[-1].get("content") or "").strip().splitlines()
            print(f"  последний разбор: {head[0][:88] if head else ''}")
        print("  вердикт: " + ("план схлопнулся, нарядов не было" if TASK3_OK
                               else "наряды были, разбора нет или задача ещё в работе"))
elif TASK3_ISSUE:
    print("\nпропуск проверки по платформе: CLI Multica недоступен")
else:
    print("\nТрек B: впишите ключ задачи в TASK3_ISSUE, чтобы проверка прошла по платформе")

## Задача 4. Сломать намеренно: крафт в ту же ступень, что и добыча

Ступени — не украшение плана. Уберите барьер, и рой сломается предсказуемым образом. Это стоит
увидеть своими глазами один раз, чтобы потом узнавать симптом за секунду.

**Трек B.** Раздайте наряды руками, положив крафт в **ту же** ступень, что и добычу:

```bash
multica issue create --title "Добыть 9 wood и сдать на склад" \
  --parent <ключ цели> --stage 1 --assignee worker-luke --status todo --description-file ./wood.md
multica issue create --title "Скрафтить 3 axe_handle, затем 3 spear" \
  --parent <ключ цели> --stage 1 --assignee worker-han --status todo --description-file ./craft.md
```

Крафтер проснётся сразу, честно вызовет `craft` и получит `not_enough_resources`. Впишите ключ его
наряда в `TASK4_ISSUE` — ячейка найдёт код ошибки в комментариях, то есть проверит по следу, а не по
вашему рассказу.

**Оба трека.** Объясните в `TASK4_REASON`, какая именно инварианта нарушена. Ответ «не хватило
ресурсов» не годится — это симптом. Нужна причина: какие две зоны хранения есть в мире, кто из них
пишет и кто читает, и почему между добычей и крафтом обязан стоять барьер.

In [ ]:
# Задача 4 — намеренная поломка. Проверка ищет след ошибки в комментариях наряда.
TASK4_ISSUE = ""     # трек B: ключ наряда на крафт, который упёрся в not_enough_resources

TASK4_REASON = """
"""            # оба трека: какая инварианта нарушена и почему барьер обязателен

TASK4_TRACE_OK = False
if TASK4_ISSUE and HAVE_MULTICA:
    comments = mc_json("issue", "comment", "list", TASK4_ISSUE) or []
    hits = [c for c in comments if "not_enough_resources" in (c.get("content") or "")]
    TASK4_TRACE_OK = bool(hits)
    print(f"{TASK4_ISSUE}: комментариев {len(comments)}, "
          f"со следом not_enough_resources — {len(hits)}")
    for c in hits[:2]:
        first = (c.get("content") or "").strip().splitlines()[0][:96]
        who = AGENT_NAMES.get(c.get("author_id"), c.get("author_type", "?"))
        print(f"  {who:<14} {first}")
    if not hits:
        print("  следа нет: либо наряд не тот, либо крафтер до craft не дошёл")
elif TASK4_ISSUE:
    print("пропуск проверки по платформе: CLI Multica недоступен")
else:
    print("Трек B: впишите ключ наряда на крафт в TASK4_ISSUE")

KEYWORDS = {
    "общий склад": ("склад", "stored"),
    "личный рюкзак": ("рюкзак", "inventory"),
    "барьер ступеней": ("ступен", "барьер"),
}
found = {k: any(w in TASK4_REASON.lower() for w in variants) for k, variants in KEYWORDS.items()}
TASK4_REASON_OK = bool(TASK4_REASON.strip()) and all(found.values())

print("\nВаше объяснение:")
if not TASK4_REASON.strip():
    print("  пусто")
else:
    for line in TASK4_REASON.strip().splitlines():
        print("  " + line)
print("\nчего касается объяснение:")
for topic, ok in found.items():
    print(f"  {'есть' if ok else 'нет '}  {topic}")
print(f"\nЗадача 4: разбор {'зачтён' if TASK4_REASON_OK else 'не зачтён'}, "
      f"след в платформе {'найден' if TASK4_TRACE_OK else 'не найден'}")

## Зачёт: критерии приёма

Всё проверяется без преподавателя. Ячейка ниже собирает результаты предыдущих шагов в одну табличку:
что выполнено, что нет, что не относится к вашему треку.

Отдельная строка — про секреты. Ячейка читает **сам файл ноутбука** и убеждается, что под-токен в
него не просочился: ни в код, ни в сохранённый вывод. Если строка не сходится — вычистите вывод
ячеек, сохраните файл заново и перезапустите зачёт. Это не формальность: сдаваемый ноутбук уходит в
публичный репозиторий.

In [ ]:
# Зачёт — итоговая табличка критериев приёма.
TRACK = "A"      # поставьте "B", если делали живой прогон на Multica

g = globals().get
token_leak = None
if TOKEN and NB_PATH.is_file():
    token_leak = TOKEN in NB_PATH.read_text(encoding="utf-8", errors="replace")

rows = [
    ("Префлайт прогнан, чего нет — названо честно", bool(g("checks")), "оба"),
    ("Задача 1: карта заготовки сошлась (файлы + будильники)", g("TASK1_OK", False), "оба"),
    ("Планировщик: видно и схлопывание, и полный разворот",
     g("TASK3_PLANNER_OK", False), "оба"),
    ("Задача 2 (A): расширение расписано шагами и файлами", g("TASK2_OK_A", False), "A"),
    ("Задача 4: нарушенная инварианта названа своими словами", g("TASK4_REASON_OK", False), "оба"),
    ("Мир прочитан по MCP, а не по отчётам агентов", g("MCP") is not None, "B"),
    ("Дерево задач прочитано из платформы, ступени видны", g("TREE_ROOT") is not None, "B"),
    ("Задача 2 (B): 4-й помощник на том же тексте инструкций", g("TASK2_OK_B", False), "B"),
    ("Задача 3 (B): цель, покрытая складом, схлопнула план", g("TASK3_OK", False), "B"),
    ("Задача 4 (B): not_enough_resources получен на живом рое", g("TASK4_TRACE_OK", False), "B"),
    ("Токена нет в файле ноутбука", token_leak is not True, "оба"),
]

print(f"Трек {TRACK}\n")
print(f"{'критерий':<62} {'трек':<6} итог")
print("-" * 92)
required = done = 0
for name, ok, track in rows:
    if track in ("оба", TRACK):
        required += 1
        done += 1 if ok else 0
        verdict = "выполнено" if ok else "не выполнено"
    else:
        verdict = "не мой трек"
    print(f"{name:<62} {track:<6} {verdict}")

print("-" * 92)
print(f"Выполнено {done} из {required} по треку {TRACK}")
if token_leak:
    print("\nВнимание: под-токен нашёлся в файле ноутбука. Очистите вывод и сохраните заново.")
if done == required:
    print("\nДомашка закрыта. Сдавайте вывод этой ячейки.")
else:
    print("\nОсталось добрать строки со статусом «не выполнено».")

if g("MCP") is not None:
    MCP.close()
    MCP = None
    print("\nMCP-подпроцесс закрыт.")

## Что сдавать

Артефакт домашки — **вывод проверочных ячеек**: склад из мира (шаг 3), дерево задач из платформы
(шаг 4) и заполненная табличка критериев (зачёт). Плюс короткий разбор «кто кого будил» своими
словами: какое событие подняло старосту, какое — каждого помощника и какое событие вы ожидали, а его
не случилось.

Ноутбук перед сдачей — **без токена в выводе**. Строка «Токена нет в файле ноутбука» в табличке
именно про это.

В чат курса: `[Модуль 16.6.5, ДЗ] {ссылка}`.

## Куда смотреть дальше

Модуль 16.7 решает ровно ту же задачу — «сделать копьё чужими руками», — но **без платформы**:
координация уезжает в очередь поручений внутри игры, а агенты становятся четырьмя asyncio-петлями в
одном питон-процессе. Пройдите оба и сравните, что именно платформа берёт на себя: очередь как шину,
пробуждение по событию и наблюдаемость. Всё остальное — ваши инструкции и ваш мир — остаётся вашей
работой в любом случае.